# **🏠 Predict House Prices**

---

# Regression Modelling

## Objectives

- Fit and evaluate regression models to predict house `SalePrice`.
- Identify a model that provides suitable predictive performance on unseen data.

## Inputs

- `outputs/datasets/cleaned/TrainSetCleaned.csv`
- `outputs/datasets/cleaned/TestSetCleaned.csv`
- Feature engineering decisions identified in the previous notebook.

## Outputs

- Train set (features and target).
- Test set (features and target).
- ML pipeline to predict house `SalePrice`.
- Feature importance plot.

## Additional Comments

This notebook forms part of the **Modelling and Evaluation** stages of the CRISP-DM process.

---

## ⚙️ 2. Set Up the Notebook

### Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

---

## 📥 3. Load Data

Load the cleaned training and test datasets produced during the data cleaning stage.

In [ ]:
import numpy as np
import pandas as pd

TrainSet = pd.read_csv(
    "outputs/datasets/cleaned/TrainSetCleaned.csv"
)

TestSet = pd.read_csv(
    "outputs/datasets/cleaned/TestSetCleaned.csv"
)

print(f"Train Set shape: {TrainSet.shape}")
print(f"Test Set shape: {TestSet.shape}")

---

## 🔧 4. Define the ML Pipeline

### Feature Engineering and Modelling Pipeline

A pipeline is created to apply the feature engineering steps identified in the
previous notebook.

Specifically, the pipeline will:

- Encode the categorical features (`OrdinalEncoder` and `OneHotEncoder`)
- Apply numerical transformations (`LogTransformer` and `PowerTransformer`)
- Scale the features (`StandardScaler`)
- Select features using `SelectFromModel`
- Fit a regression model

In [ ]:
from sklearn.pipeline import Pipeline

# Feature Engineering
from feature_engine.encoding import OrdinalEncoder, OneHotEncoder
from feature_engine import transformation as vt

# Feature Scaling
from sklearn.preprocessing import StandardScaler

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# ML Algorithms
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor
)
from xgboost import XGBRegressor


def PipelineOptimization(model):
    """
    Creates a regression pipeline using the feature engineering decisions
    identified in the Feature Engineering notebook.
    """
    pipeline_base = Pipeline([

        # CATEGORICAL ENCODING
        ("OrdinalCategoricalEncoder", OrdinalEncoder(
    encoding_method='arbitrary',
    variables=[
        'BsmtExposure',
        'GarageFinish',
        'KitchenQual'
    ],
    unseen='encode'
)),

        ("OneHotCategoricalEncoder", OneHotEncoder(
            variables=['BsmtFinType1'],
            drop_last=False
        )),

        # NUMERICAL TRANSFORMATIONS
        ("LogTransformer", vt.LogTransformer(
            variables=[
                '1stFlrSF',
                'GrLivArea',
                'LotArea',
                'LotFrontage'
            ]
        )),

        ("PowerTransformer", vt.PowerTransformer(
            variables=[
                'BsmtUnfSF',
                'TotalBsmtSF'
            ]
        )),

        # FEATURE SCALING
        ("feat_scaling", StandardScaler()),

        # FEATURE SELECTION
        ("feat_selection", SelectFromModel(model)),

        # MODEL
        ("model", model),

    ])

    return pipeline_base

### Hyperparameter Optimisation Class

The `HyperparameterOptimizationSearch` class is adapted from the Code Institute Predictive Analytics course material.

The class works as follows:

- `__init__` receives the models and their hyperparameters.
- `fit` tests each model and its hyperparameters using `GridSearchCV`.
- `score_summary` summarises the performance of each model.

This allows us to compare the models and identify the best-performing configuration.

In [ ]:
from sklearn.model_selection import GridSearchCV


class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")

            model = PipelineOptimization(self.models[key])
            params = self.params[key]

            gs = GridSearchCV(
                model,
                params,
                cv=cv,
                n_jobs=n_jobs,
                verbose=verbose,
                scoring=scoring
            )

            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):

        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []

        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []

            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)

            for p, s in zip(params, all_scores):
                rows.append(row(k, s, p))

        df = pd.concat(rows, axis=1).T.sort_values(
            [sort_by], ascending=False
        )

        columns = [
            'estimator',
            'min_score',
            'mean_score',
            'max_score',
            'std_score'
        ]

        columns = columns + [
            c for c in df.columns if c not in columns
        ]

        return df[columns], self.grid_searches

---

## ✂️ 5. Prepare Train and Test Sets

The training and test sets were created during the data cleaning stage to prevent data leakage.

The predictor variables and target variable `SalePrice` are now separated for modelling.

In [ ]:
X_train = TrainSet.drop(columns=['SalePrice'])
y_train = TrainSet['SalePrice']

X_test = TestSet.drop(columns=['SalePrice'])
y_test = TestSet['SalePrice']

print(
    "* Train set:", X_train.shape, y_train.shape,
    "\n* Test set:", X_test.shape, y_test.shape
)

### Train Set Target Distribution

The distribution of the training target `SalePrice` is inspected before modelling.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

sns.histplot(y_train, kde=True)
plt.title("Train Set SalePrice Distribution")
plt.xlabel("SalePrice")
plt.show()

### Target Distribution Interpretation

- `SalePrice` is positively skewed, with most houses priced between roughly 100,000 and 250,000 and fewer houses at the higher end.

- No transformation will currently be applied to `SalePrice` before modelling.

---

## 🔎 6. Quick Model Search

The first step is to compare a range of regression models using their default
hyperparameters.

### Define Models

We now define the regression models to compare, using their default hyperparameters.

In [ ]:
models_quick_search = {
    "LinearRegression": LinearRegression(),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=0),
    "RandomForestRegressor": RandomForestRegressor(random_state=0),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
    "AdaBoostRegressor": AdaBoostRegressor(random_state=0),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=0),
    "XGBRegressor": XGBRegressor(random_state=0),
}

params_quick_search = {
    "LinearRegression": {},
    "DecisionTreeRegressor": {},
    "RandomForestRegressor": {},
    "ExtraTreesRegressor": {},
    "AdaBoostRegressor": {},
    "GradientBoostingRegressor": {},
    "XGBRegressor": {},
}

### Model Performance Requirement

The project requirement is to achieve an **R² score of at least 0.75 on both
the training and test sets**.

We now run the quick search using the training data.

The models are compared using their default hyperparameters, with `R²` used to measure performance and 5-fold cross-validation (`cv=5`).

In [ ]:
search = HyperparameterOptimizationSearch(
    models=models_quick_search,
    params=params_quick_search
)

search.fit(
    X_train,
    y_train,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

### Quick Search Results

LinearRegression achieved the highest mean cross-validation R² score (0.822), followed by ExtraTreesRegressor (0.814) and RandomForestRegressor (0.804).

LinearRegression and ExtraTreesRegressor were selected for further evaluation.

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(
    sort_by='mean_score'
)

grid_search_summary

| **Estimator**             | **Min Score** | **Mean Score** | **Max Score** | **Std Score** |
| ------------------------- | ------------: | -------------: | ------------: | ------------: |
| LinearRegression          |         0.750 |          0.822 |         0.862 |         0.043 |
| ExtraTreesRegressor       |         0.667 |          0.814 |         0.874 |         0.075 |
| RandomForestRegressor     |         0.696 |          0.804 |         0.853 |         0.060 |
| GradientBoostingRegressor |         0.612 |          0.775 |         0.852 |         0.091 |
| AdaBoostRegressor         |         0.679 |          0.757 |         0.806 |         0.042 |
| XGBRegressor              |         0.516 |          0.677 |         0.783 |         0.105 |
| DecisionTreeRegressor     |         0.495 |          0.608 |         0.703 |         0.080 |

---

## ⚙️ 7. Hyperparameter Optimisation

Following the quick model search, we now perform a more extensive search on the most suitable models.

The aim is to find the best hyperparameter configuration using `GridSearchCV`.

### Define Models and Hyperparameters

`LinearRegression` and `ExtraTreesRegressor` achieved the strongest
performance during the quick search and are therefore selected for further
optimisation.

The hyperparameter settings are based on the recommendations provided in the Code Institute Predictive Analytics course material.

In [ ]:
models_search = {
    "LinearRegression": LinearRegression(),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
}

params_search = {
    "LinearRegression": {},

    "ExtraTreesRegressor": {
        "model__n_estimators": [100, 50, 150],
        "model__max_depth": [None, 3, 15],
        "model__min_samples_split": [2, 50],
        "model__min_samples_leaf": [1, 50],
    },
}

### Run Extensive Search

We now run the hyperparameter search using `R²` as the scoring metric and 5-fold cross-validation.

In [ ]:
search = HyperparameterOptimizationSearch(
    models=models_search,
    params=params_search
)

search.fit(
    X_train,
    y_train,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

### Hyperparameter Search Results

We summarise the results to identify the best-performing model and hyperparameter configuration.

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(
    sort_by='mean_score'
)

grid_search_summary

### Select Best Model and Parameters

Although LinearRegression achieved a slightly higher mean score, ExtraTreesRegressor demonstrated competitive performance and a lower standard deviation across the cross-validation folds. Therefore we will go with ExtraTrees Regressor.

In [ ]:
best_model = "ExtraTreesRegressor"
best_model

In [ ]:
best_parameters = grid_search_pipelines[best_model].best_params_
best_parameters

In [ ]:
pipeline_reg = grid_search_pipelines[best_model].best_estimator_
pipeline_reg

### Hyperparameter Optimisation Interpretation

LinearRegression achieved the highest mean cross-validation R² score of 0.822, closely followed by ExtraTreesRegressor with 0.818.

The best ExtraTreesRegressor configuration used `max_depth=15`, `min_samples_split=50`, `min_samples_leaf=1` and `n_estimators=50`.

Although LinearRegression achieved a slightly higher mean score, ExtraTreesRegressor had a lower standard deviation across the cross-validation folds.

ExtraTreesRegressor was therefore retained for further feature importance analysis and additional hyperparameter optimisation.

---

## 📊 8. Feature Importance

Next, we will identify the most important features using `feature_importances_`.

In [ ]:
# After feature engineering, the feature space may change.
# Get the column names after the feature engineering steps.

data_cleaning_feat_eng_steps = 4

columns_after_data_cleaning_feat_eng = (
    Pipeline(pipeline_reg.steps[:data_cleaning_feat_eng_steps])
    .transform(X_train)
    .columns
)

# Get the features retained by SelectFromModel
best_features = columns_after_data_cleaning_feat_eng[
    pipeline_reg['feat_selection'].get_support()
].to_list()


# Create DataFrame to display feature importance
df_feature_importance = (
    pd.DataFrame(data={
        'Feature': best_features,
        'Importance': pipeline_reg['model'].feature_importances_
    })
    .sort_values(by='Importance', ascending=False)
)


# Most important features statement and plot
print(
    f"* These are the {len(best_features)} most important features "
    f"in descending order. The model was trained on them: "
    f"\n{df_feature_importance['Feature'].to_list()}"
)

df_feature_importance.plot(
    kind='bar',
    x='Feature',
    y='Importance'
)

plt.show()

### Interpretation of Results

The feature-selection stage selected seven features using `SelectFromModel`:

- `OverallQual`
- `GrLivArea`
- `YearBuilt`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `YearRemodAdd`

`OverallQual` was the most important feature by a considerable margin.


---

## 📈 9. Model Evaluation

The model is evaluated using:

- R² Score
- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)

This will be done using our adapted Code Institute function.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error 
import numpy as np


def regression_performance(X_train, y_train, X_test, y_test, pipeline):
    print("Model Evaluation \n")
    print("* Train Set")
    regression_evaluation(X_train, y_train, pipeline)
    print("* Test Set")
    regression_evaluation(X_test, y_test, pipeline)


def regression_evaluation(X, y, pipeline):
    prediction = pipeline.predict(X)

    print('R2 Score:', r2_score(y, prediction).round(3))  
    print('Mean Absolute Error:', mean_absolute_error(y, prediction).round(3))  
    print('Mean Squared Error:', mean_squared_error(y, prediction).round(3))  
    print('Root Mean Squared Error:',
          np.sqrt(mean_squared_error(y, prediction)).round(3))
    print("\n")


def regression_evaluation_plots(
    X_train, y_train, X_test, y_test, pipeline, alpha_scatter=0.5
):
    pred_train = pipeline.predict(X_train)
    pred_test = pipeline.predict(X_test)

    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

    sns.scatterplot(
        x=y_train, y=pred_train,
        alpha=alpha_scatter, ax=axes[0]
    )
    sns.lineplot(x=y_train, y=y_train, color='red', ax=axes[0])
    axes[0].set_xlabel("Actual")
    axes[0].set_ylabel("Predictions")
    axes[0].set_title("Train Set")

    sns.scatterplot(
        x=y_test, y=pred_test,
        alpha=alpha_scatter, ax=axes[1]
    )
    sns.lineplot(x=y_test, y=y_test, color='red', ax=axes[1])
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Predictions")
    axes[1].set_title("Test Set")

    plt.show()

### Train and Test Performance

We evaluate the performance of both the train and test set. This is done by comparing the actual sales price to the predicted sales price.

In [ ]:
regression_performance(
    X_train, y_train,
    X_test, y_test,
    pipeline_reg
)

regression_evaluation_plots(
    X_train, y_train,
    X_test, y_test,
    pipeline_reg,
    alpha_scatter=0.5
)

### Model Evaluation Interpretation

The table below summarises the performance of the selected ExtraTreesRegressor model on the training and test sets.

| Metric | Train Set | Test Set |
|---|---:|---:|
| R² Score | 0.848 | 0.778 |
| Mean Absolute Error (MAE) | 19,080.564 | 21,176.606 |

- **R²:** The model achieved 0.848 on the training set and 0.778 on the test set, meeting the client's minimum requirement of 0.75.
- **Prediction error:** The average absolute error was approximately 19,081 on the training set and 21,177 on the test set. The higher RMSE indicates some larger prediction errors.
- **Prediction plots:** The model performs better on lower-priced properties but is less accurate for some expensive houses.

### 9.1 Linear Regression Comparison

Just as a quick experiment I want to check what R2 result Linear regression would have achieved. I will run this check because Linear Regression achieved the highest mean cross-validation R² score (0.822) during the initial model search.

In [ ]:
# Retrieve the optimised LinearRegression pipeline from Section 7
linear_pipeline = grid_search_pipelines[
    "LinearRegression"
].best_estimator_

# Evaluate on the original full-feature datasets
regression_performance(
    TrainSet.drop(columns=['SalePrice']),
    y_train,
    TestSet.drop(columns=['SalePrice']),
    y_test,
    linear_pipeline
)

#### Linear Regression — Evaluation

LinearRegression was evaluated using the same training and test datasets as the original model.

| Metric | Train Set | Test Set |
|---|---:|---:|
| R² Score | 0.835 | 0.688 |
| Mean Absolute Error (MAE) | $20,669 | $24,663 |
| Root Mean Squared Error (RMSE) | $31,861 | $46,410 |

Linear regression achieved the client´s minimum requirement on the train dataset but did not meet the client minimum requirement on the test set.

**Decision:** Do not retain LinearRegression for further optimisation. Continue with ExtraTreesRegressor, which met the client's minimum R² requirement on both datasets.

---

## 🧪 10. Further Model Experiments

We have already met the client's minimum requirement of an R² score of at least 0.75 on both the training and test sets. However, the model's performance could potentially be improved. We will run two experiments to investigate this.

- **SmartCorrelatedSelection:** Test an alternative feature selection method.
- **Additional Hyperparameter Optimisation:** Explore different hyperparameter values to improve model performance.

The results will be compared with the original model to determine which approach should be used for the final pipeline.

### 10.1 SmartCorrelatedSelection Experiment

SelectFromModel was replaced with SmartCorrelatedSelection to investigate whether this improves predictive performance.

The same ExtraTreesRegressor hyperparameters are used to evaluate the alternative approach.

In [ ]:
# Use the full cleaned datasets for this experiment
X_train_smart = TrainSet.drop(columns=['SalePrice'])
X_test_smart = TestSet.drop(columns=['SalePrice'])

print("Train shape:", X_train_smart.shape)
print("Test shape:", X_test_smart.shape)

In [ ]:
from feature_engine.selection import SmartCorrelatedSelection

def PipelineSmartCorrelation(model):

    pipeline = Pipeline([

        # CATEGORICAL ENCODING
        ("OrdinalCategoricalEncoder", OrdinalEncoder(
            encoding_method='arbitrary',
            variables=[
                'BsmtExposure',
                'GarageFinish',
                'KitchenQual'
            ],
            unseen='encode'
        )),

        ("OneHotCategoricalEncoder", OneHotEncoder(
            variables=['BsmtFinType1'],
            drop_last=False
        )),

        # NUMERICAL TRANSFORMATIONS
        ("LogTransformer", vt.LogTransformer(
            variables=[
                '1stFlrSF',
                'GrLivArea',
                'LotArea',
                'LotFrontage'
            ]
        )),

        ("PowerTransformer", vt.PowerTransformer(
            variables=[
                'BsmtUnfSF',
                'TotalBsmtSF'
            ]
        )),

        # CORRELATED FEATURE SELECTION
        ("feat_selection", SmartCorrelatedSelection(
            variables=None,
            method='spearman',
            threshold=0.6,
            selection_method='variance'
        )),

        # FEATURE SCALING
        ("feat_scaling", StandardScaler()),

        # MODEL
        ("model", model)

    ])

    return pipeline

In [ ]:
# Create the experimental pipeline
smart_corr_pipeline = PipelineSmartCorrelation(
    ExtraTreesRegressor(
        max_depth=15,
        min_samples_split=50,
        min_samples_leaf=1,
        n_estimators=50,
        random_state=0
    )
)

# Fit the experimental model
smart_corr_pipeline.fit(X_train_smart, y_train)

# Evaluate model performance
regression_performance(
    X_train_smart,
    y_train,
    X_test_smart,
    y_test,
    smart_corr_pipeline
)


#### SmartCorrelatedSelection — Evaluation

The experimental pipeline was evaluated using the same training and test datasets as the original model.

| Metric | Existing Model | SmartCorrelatedSelection |
|---|---:|---:|
| Train R² | 0.848 | 0.849 |
| Test R² | 0.778 | 0.759 |
| Test MAE | $21,177 | $22,782 |

Although the experimental model met the client's minimum R² requirement of 0.75 on both datasets, it did not improve performance on unseen data.

Test R² decreased from 0.778 to 0.759, while test MAE increased from approximately 21,177 to 22,782.

**Decision:** Retain `SelectFromModel` rather than `SmartCorrelatedSelection` in the final pipeline.


### 10.2 Additional Hyperparameter Optimisation

An extended GridSearchCV was performed on `ExtraTreesRegressor` using six hyperparameters, each with three values.

| Hyperparameter | Values | Rationale |
|---|---|---|
| `n_estimators` | 50, 100, 150 | Number of trees |
| `max_depth` | 10, 20, None | Control tree depth |
| `min_samples_split` | 5, 10, 20 | Control splitting |
| `min_samples_leaf` | 1, 2, 4 | Control leaf size |
| `max_features` | 0.6, 0.8, 1.0 | Control features per split |
| `min_impurity_decrease` | 0.0, 0.01, 0.1 | Control minimum split improvement |

Three grouped grids were used to limit processing time to 135 fits. Five-fold cross-validation was used to select the best model.

#### Test Different Hyperparameters

In [ ]:
models_tuning = {
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
}

params_tuning = {
    "ExtraTreesRegressor": [
        {
            "model__n_estimators": [50, 100, 150],
            "model__max_depth": [10, 20, None],
            "model__min_samples_split": [5],
            "model__min_samples_leaf": [4],
            "model__max_features": [1.0],
            "model__min_impurity_decrease": [0.0],
        },
        {
            "model__n_estimators": [100],
            "model__max_depth": [None],
            "model__min_samples_split": [5, 10, 20],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [1.0],
            "model__min_impurity_decrease": [0.0],
        },
        {
            "model__n_estimators": [100],
            "model__max_depth": [None],
            "model__min_samples_split": [5],
            "model__min_samples_leaf": [4],
            "model__max_features": [0.6, 0.8, 1.0],
            "model__min_impurity_decrease": [0.0, 0.01, 0.1],
        },
    ],
}

#### Run Extensive Search

In [ ]:
# Use full feature sets for this experiment
X_train_tuning = TrainSet.drop(columns=["SalePrice"])
X_test_tuning = TestSet.drop(columns=["SalePrice"])

# Create an independent search
tuning_search = HyperparameterOptimizationSearch(
    models=models_tuning,
    params=params_tuning
)

# Fit using training data only
tuning_search.fit(
    X_train_tuning,
    y_train,
    scoring="r2",
    n_jobs=-1,
    cv=5
)

In [ ]:
tuning_summary, tuning_pipelines = tuning_search.score_summary(
    sort_by="mean_score"
)

tuning_summary

In [ ]:
tuned_model = tuning_summary.iloc[0, 0]
tuned_model

In [ ]:
tuned_parameters = tuning_pipelines[tuned_model].best_params_
tuned_parameters

The additional hyperparameter search identified the following best configuration for `ExtraTreesRegressor`:

- `n_estimators`: 50
- `max_depth`: 20 (maximum tree depth)
- `min_samples_split`: 5
- `min_samples_leaf`: 4
- `max_features`: 1.0 (all features considered at each split)
- `min_impurity_decrease`: 0.0 (no minimum impurity reduction required)

These parameters were selected based on the highest mean cross-validation R² score.

In [ ]:
tuned_pipeline = tuning_pipelines[tuned_model].best_estimator_
tuned_pipeline

In [ ]:
regression_performance(
    X_train_tuning,
    y_train,
    X_test_tuning,
    y_test,
    tuned_pipeline
)

regression_evaluation_plots(
    X_train_tuning,
    y_train,
    X_test_tuning,
    y_test,
    tuned_pipeline,
    alpha_scatter=0.5
)


#### Original vs Tuned ExtraTreesRegressor

| Metric | Original Train | Original Test | Tuned Train | Tuned Test |
|---|---:|---:|---:|---:|
| R² Score | 0.848 | 0.778 | 0.912 | 0.808 |
| Mean Absolute Error (MAE) | 19,080.564 | 21,176.606 | 14,359.638 | 20,214.550 |

The additional hyperparameter optimisation improved the test R² score from 0.778 to 0.808 and reduced the test MAE from approximately 21,177 to 20,215.

Both training and test R² scores exceed the client's minimum requirement of 0.75. 

**Decision:** Retain the optimised ExtraTreesRegressor for the final regression pipeline with the optimised hyperparameters.

---

## 🔁 11. Refit with the Most Important Features

From our previous analyses, we have selected ExtraTreesRegressor, optimised its hyperparameters, and identified the seven most important features.

| Component | Selected result |
|---|---|
| **Selected algorithm** | `ExtraTreesRegressor` |
| **Optimised hyperparameters (Section 10)** | `n_estimators=50`, `max_depth=20`, `min_samples_split=5`, `min_samples_leaf=4`, `max_features=1.0`, `min_impurity_decrease=0.0` |
| **Selected features (Section 8)** | `OverallQual`, `GrLivArea`, `YearBuilt`, `GarageArea`, `TotalBsmtSF`, `1stFlrSF`, `YearRemodAdd` |

We will refit the pipeline using these seven features and the optimised hyperparameters.

### Prepare Train and Test Sets

The training and test sets were already created during the data cleaning stage.

We now prepare the original predictor and target variables before filtering to the most important features.

In [ ]:
X_train = TrainSet.drop(columns=['SalePrice'])
y_train = TrainSet['SalePrice']

X_test = TestSet.drop(columns=['SalePrice'])
y_test = TestSet['SalePrice']

print(
    "* Train set:", X_train.shape, y_train.shape,
    "\n* Test set:", X_test.shape, y_test.shape
)

### Filter to Selected Features

Now we subset the best features.

In [ ]:
X_train = TrainSet[best_features].copy()
X_test = TestSet[best_features].copy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Selected features:", best_features)


### Define the Simplified Pipeline

**Original Pipeline**

`20 Features` → `Feature Engineering` → `Scaling` → `Feature Selection` → `7 Features` → `Model`

**Refitted Pipeline**

`7 Features` → `Feature Engineering` → `Scaling` → `Model`

- Categorical encoders were removed because none of the seven selected features are categorical.
- `SelectFromModel` was removed because the training and test datasets have already been filtered to the seven selected features.
- `ExtraTreesRegressor` will be refitted using the optimised hyperparameters identified in Section 10.

In [ ]:
from sklearn.pipeline import Pipeline

# Feature Engineering
from feature_engine import transformation as vt

# Feature Scaling
from sklearn.preprocessing import StandardScaler

# ML Algorithm
from sklearn.ensemble import ExtraTreesRegressor


def PipelineFinalRegression(model):
    """
    Creates the final regression pipeline using only the
    seven most important features.
    """

    pipeline_base = Pipeline([

        # NUMERICAL TRANSFORMATIONS
        ("LogTransformer", vt.LogTransformer(
            variables=[
                '1stFlrSF',
                'GrLivArea'
            ]
        )),

        ("PowerTransformer", vt.PowerTransformer(
            variables=['TotalBsmtSF']
        )),

        # FEATURE SCALING
        ("feat_scaling", StandardScaler()),

        # MODEL
        ("model", model),

    ])

    return pipeline_base

### Refit the Best Model

We now refit ExtraTreesRegressor using the optimised hyperparameters identified in Section 10 and the seven selected features.

In [ ]:
best_pipeline = PipelineFinalRegression(
    ExtraTreesRegressor(
        n_estimators=50,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=4,
        max_features=1.0,
        min_impurity_decrease=0.0,
        random_state=0
    )
)

best_pipeline.fit(X_train, y_train)

best_pipeline

### Evaluate the Refitted Model

Finally, we evaluate the refitted pipeline to confirm that using only the selected features has not reduced model performance.

In [ ]:
regression_performance(
    X_train,
    y_train,
    X_test,
    y_test,
    best_pipeline
)

regression_evaluation_plots(
    X_train,
    y_train,
    X_test,
    y_test,
    best_pipeline,
    alpha_scatter=0.5
)

### Refit Results

- The model was refitted using the seven features identified during feature importance.
- The feature selection stage was removed as the seven features are supplied directly to the model.
- The optimised hyperparameters from Section 10 were retained.
- The refitted model achieved an R² score of **0.912 on the training set** and **0.808 on the test set**, exceeding the client requirement of 0.75.
- These results match Section 10, confirming that simplifying the pipeline did not reduce model performance.

---

## 💾 12. Save Final Model and Outputs

The following will now be saved:
- Train set
- Test set
- Final regression pipeline
- Feature importance plot

In [ ]:
import joblib
import os

version = 'v1'
file_path = f'outputs/ml_pipeline/predict_house_prices/{version}'

try:
    os.makedirs(name=file_path)
except Exception as e:
    print(e)

### Train Set: features and target

Save the final train set.

In [ ]:
X_train.head()

In [ ]:
X_train.to_csv(f"{file_path}/X_train.csv", index=False)

In [ ]:
y_train

In [ ]:
y_train.to_csv(f"{file_path}/y_train.csv", index=False)

### Test Set: features and target

Save the final test set.

In [ ]:
X_test.head()

In [ ]:
X_test.to_csv(f"{file_path}/X_test.csv", index=False)

In [ ]:
y_test

In [ ]:
y_test.to_csv(f"{file_path}/y_test.csv", index=False)

### Modelling Pipeline

ML pipeline for predicting house sale prices.

In [ ]:
best_pipeline

In [ ]:
joblib.dump(
    value=best_pipeline,
    filename=f"{file_path}/reg_pipeline.pkl"
)

### Feature Importance Plot

Save the feature importance plot for use in the dashboard.

In [ ]:
df_final_feature_importance = (
    pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': best_pipeline['model'].feature_importances_
    })
    .sort_values(by='Importance', ascending=False)
)

df_final_feature_importance.plot(
    kind='bar',
    x='Feature',
    y='Importance'
)

plt.savefig(
    f'{file_path}/features_importance.png',
    bbox_inches='tight'
)

plt.show()

---